# Environment Preparation

In [ ]:
!pip install -q -U peft transformers accelerate hqq datasets


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 60.0 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 33.8 MB/s eta 0:00:00


In [ ]:
import torch
from hqq.models.hf.base import AutoHQQHFModel
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset


2025-12-23 21:12:24.797991: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766524344.929510      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766524344.966598      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766524345.279339      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766524345.279386      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766524345.279389      55 computation_placer.cc:177] computation placer alr

In [ ]:
MAX_STEPS = 120 
BATCH_SIZE = 4
GRAD_ACCUM = 4
MAX_LEN = 256


# Load Model

In [ ]:
MODEL_ID = "Neuro-Poplar/qwen3-8b-hqq-4bit"
model = AutoHQQHFModel.from_quantized(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

# Prepare for Tuning

In [ ]:
model.gradient_checkpointing_enable()
model.enable_input_require_grads()


In [ ]:
lora_config = LoraConfig(
    r=8, 
    lora_alpha=16, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 7,667,712 || all params: 4,725,519,360 || trainable%: 0.1623


# Load Data

In [ ]:
dataset = load_dataset("cais/mmlu", "all", split="auxiliary_train")
dataset = dataset.shuffle(seed=42).select(range(1600))


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

all/dev-00000-of-00001.parquet:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

all/auxiliary_train-00000-of-00001.parqu(…):   0%|          | 0.00/47.5M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

In [ ]:
def tokenize_function(examples):
    texts = [f"Question: {q}\nAnswer: {chr(65 + a)}" for q, a in zip(examples['question'], examples['answer'])]
    return tokenizer(texts, truncation=True, padding="max_length", max_length=MAX_LEN)


In [ ]:
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

# Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen3-8b-hqq-4bit-finetuned",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=12,
    max_steps=MAX_STEPS,
    learning_rate=3e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="no",
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,2.722300
20,2.325200
30,2.246000
40,2.027800
50,2.076400
60,2.100500
70,2.068200
80,2.084900
90,2.056100
100,2.049500


TrainOutput(global_step=120, training_loss=2.1522576332092287, metrics={'train_runtime': 3792.5134, 'train_samples_per_second': 0.506, 'train_steps_per_second': 0.032, 'total_flos': 1.210079823003648e+16, 'train_loss': 2.1522576332092287, 'epoch': 1.2})

In [ ]:
model.save_pretrained("./qwen3-8b-lora-adapter")
tokenizer.save_pretrained("./qwen3-8b-lora-adapter")


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in models--Neuro-Poplar--qwen3-8b-hqq-4bit/snapshots/bd4f480d86c54956799a6a14b8e76ffa9f3cf85f/config.json - will assume that the vocabulary was not modified.
  warnings.warn(


('./qwen3-8b-lora-adapter/tokenizer_config.json',
 './qwen3-8b-lora-adapter/special_tokens_map.json',
 './qwen3-8b-lora-adapter/chat_template.jinja',
 './qwen3-8b-lora-adapter/vocab.json',
 './qwen3-8b-lora-adapter/merges.txt',
 './qwen3-8b-lora-adapter/added_tokens.json',
 './qwen3-8b-lora-adapter/tokenizer.json')

# Evaluation

In [ ]:
def evaluate_mmlu_detailed(model, tokenizer, percentage=0.2):
    """Evaluate model performance on MMLU dataset. 
    Returns mean accuracy and accuracy by each subset in DataFrame format."""
    dataset = load_dataset("cais/mmlu", "all", split="test")
    df = dataset.to_pandas()
    data_df = df.groupby('subject', group_keys=False).apply(
        lambda x: x.sample(frac=percentage, random_state=42)
    )
    
    model.eval()
    choices = ['A', 'B', 'C', 'D']

    subjects = data_df['subject'].unique()
    subject_stats = {sub: {'correct': 0, 'total': 0} for sub in subjects}
    
    choice_ids = [tokenizer.encode(c, add_special_tokens=False)[-1] for c in choices]
    prompt_template = "Question: {question}\nChoices:\nA. {a}\nB. {b}\nC. {c}\nD. {d}\nAnswer:"

    with torch.no_grad():
        for _, row in tqdm(data_df.iterrows(), total=len(data_df), desc="Evaluating"):
            prompt = prompt_template.format(
                question=row['question'],
                a=row['choices'][0], b=row['choices'][1],
                c=row['choices'][2], d=row['choices'][3]
            )
            
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            outputs = model(**inputs)
            last_logits = outputs.logits[0, -1, choice_ids]
            prediction = torch.argmax(last_logits).item()
            
            is_correct = (prediction == row['answer'])
            subject_stats[row['subject']]['total'] += 1
            if is_correct:
                subject_stats[row['subject']]['correct'] += 1

    results = []
    total_correct = 0
    total_samples = 0

    for sub, stats in subject_stats.items():
        acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
        results.append({'subject': sub, 'accuracy': acc, 'count': stats['total']})
        total_correct += stats['correct']
        total_samples += stats['total']
        
    df_results = pd.DataFrame(results)
    mean_accuracy = total_correct / total_samples
                
    return mean_accuracy, df_results

def plot_mmlu_comparison(df_comp):
    categories = {
        'STEM': ['abstract_algebra', 'anatomy', 'astronomy', 'college_biology', 'college_chemistry', 'college_computer_science', 'college_mathematics', 'college_physics', 'computer_security', 'conceptual_physics', 'electrical_engineering', 'elementary_mathematics', 'high_school_biology', 'high_school_chemistry', 'high_school_computer_science', 'high_school_mathematics', 'high_school_physics', 'statistics'],
        'Humanities': ['formal_logic', 'high_school_european_history', 'high_school_us_history', 'high_school_world_history', 'international_law', 'jurisprudence', 'logical_fallacies', 'moral_dispute', 'moral_scenarios', 'philosophy', 'prehistory', 'professional_law', 'world_religions'],
        'Social Sciences': ['econometrics', 'high_school_geography', 'high_school_government_and_politics', 'high_school_macroeconomics', 'high_school_microeconomics', 'high_school_psychology', 'human_sexuality', 'human_reproduction', 'public_relations', 'sociology', 'us_foreign_policy'],
        'Other': ['business_ethics', 'clinical_knowledge', 'global_facts', 'management', 'marketing', 'medical_genetics', 'nutrition', 'professional_accounting', 'professional_medicine', 'professional_psychology', 'virology']
    }
    def get_cat(sub):
        for c, subs in categories.items():
            if sub in subs: return c
        return 'Other'

    df_comp['category'] = df_comp['subject'].apply(get_cat)
    cat_plot = df_comp.groupby('category')[['baseline_acc', 'compressed_acc']].mean()
    
    cat_plot.plot(kind='bar', figsize=(10, 5))
    plt.title("MMLU: Baseline vs Compressed by Category")
    plt.ylabel("Accuracy")
    plt.xticks(rotation=0)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig("comparison_plot.png")
    print("Plot saved as comparison_plot.png")
    plt.show()


In [ ]:
from tqdm import tqdm
import pandas as pd


In [ ]:
micro_avg, detailed_df = evaluate_mmlu_detailed(model, tokenizer, percentage=0.2)
print(f"Final Micro-Average Accuracy after Tuning: {micro_avg:.4f}")
detailed_df.to_csv("results_finetuned.csv", index=False)


/tmp/ipykernel_55/1298761614.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data_df = df.groupby('subject', group_keys=False).apply(
Evaluating: 100%|██████████| 2809/2809 [32:31<00:00,  1.44it/s]

Final Micro-Average Accuracy after Tuning: 0.7091


In [ ]:
detailed_df


,subject,accuracy,count
0,abstract_algebra,0.600000,20
1,anatomy,0.629630,27
2,astronomy,0.966667,30
3,business_ethics,0.700000,20
4,clinical_knowledge,0.867925,53
5,college_biology,0.896552,29
6,college_chemistry,0.600000,20
7,college_computer_science,0.800000,20
8,college_mathematics,0.700000,20
9,college_medicine,0.742857,35


## Save Model to HF

In [ ]:
!pip install -q huggingface_hub


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_lab_token")


In [ ]:
import json
import os

adapter_path = "./qwen3-8b-lora-adapter"
correct_model_id = "Neuro-Poplar/qwen3-8b-hqq-4bit"

config_file = os.path.join(adapter_path, "adapter_config.json")
if os.path.exists(config_file):
    with open(config_file, 'r') as f:
        config = json.load(f)
    
    config["base_model_name_or_path"] = correct_model_id
    
    with open(config_file, 'w') as f:
        json.dump(config, f, indent=4)
    print("adapter_config.json fixed!")

readme_file = os.path.join(adapter_path, "README.md")
if os.path.exists(readme_file):
    os.remove(readme_file)
    print("Incorrect README.md removed (it will be regenerated by HF Hub).")

print("Ready to upload.")


adapter_config.json fixed!
Incorrect README.md removed (it will be regenerated by HF Hub).
Ready to upload.


In [ ]:
user_secrets = UserSecretsClient()
login(user_secrets.get_secret("HF_lab_token"))

api = HfApi()
repo_id = "Neuro-Poplar/qwen3-8b-hqq-4bit-lora-adapter"

api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

api.upload_folder(
    folder_path="./qwen3-8b-lora-adapter",
    repo_id=repo_id,
    repo_type="model"
)
print(f"Success! Model uploaded to https://huggingface.co/{repo_id}")


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Success! Model uploaded to https://huggingface.co/Neuro-Poplar/qwen3-8b-hqq-4bit-lora-adapter


In [ ]:
def get_adapter_size_mb(path):
    if not os.path.exists(path):
        return 40.0
    total_size = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            if f.endswith(('.bin', '.safetensors', '.pt')):
                total_size += os.path.getsize(os.path.join(dirpath, f))
    return total_size / (1024 * 1024)


In [ ]:
get_adapter_size_mb("./qwen3-8b-lora-adapter")


29.28656768798828